Tugas Mandiri

1. Persiapan Dataset

In [10]:
import numpy as np
import pandas as pd

# Tabel 1: Target & PIC per cabang kota (tabel referensi, dibuat langsung sebagai DataFrame)
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Tabel 2: Data transaksi (disimpan sebagai CSV, lalu diunggah ke HDFS)
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/mahasiswa/tugas5/
print("Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv")
print("Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.")

Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv
Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.


In [11]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Tugas5") \
    .master("local[*]") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://localhost:9000") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("Spark siap:", spark.version)

Spark siap: 3.5.9


Membaca Data Transaksi dari HDFS

In [17]:
from pyspark.sql.functions import (
    col,
    sum as spark_sum,
    avg,
    count,
    row_number,
    rank,
    when
)

from pyspark.sql.window import Window

df_transaksi = spark.read.csv(
    "hdfs:///user/mahasiswa/tugas5/transaksi_tugas5.csv",
    header=True,
    inferSchema=True
)

df_transaksi = df_transaksi.withColumn(
    "pendapatan",
    col("unit_terjual") * col("harga_satuan")
)

df_transaksi.show(5)

+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|    600000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|    600000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|    150000|
+--------+--------------------+----------+------------+------------+----------+
only showing top 5 rows



In [13]:
df_transaksi = df_transaksi.withColumn(
    "pendapatan",
    col("unit_terjual") * col("harga_satuan")
)

df_transaksi.show(5)

+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|    600000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|    600000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|    150000|
+--------+--------------------+----------+------------+------------+----------+
only showing top 5 rows



In [14]:
df_target = spark.createDataFrame(
    pd.DataFrame(data_target_cabang)
)

df_target.show()

+----------+--------------+----------+
|      kota|target_bulanan|pic_cabang|
+----------+--------------+----------+
|  Magelang|      45000000|      Rani|
|Yogyakarta|      60000000|      Joko|
|  Semarang|      55000000|      Sari|
|      Solo|      40000000|      Bayu|
| Purworejo|      30000000|     Fitri|
+----------+--------------+----------+



Menambahkan Kolom Pendapatan

In [15]:
df_transaksi = df_transaksi.withColumn(
    "pendapatan",
    col("unit_terjual") * col("harga_satuan")
)

df_transaksi.show(5)

+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|    600000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|    600000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|    150000|
+--------+--------------------+----------+------------+------------+----------+
only showing top 5 rows



Tugas A - Join & Perbandingan Target

In [18]:
ringkasan_kota = df_transaksi.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

hasil_A = ringkasan_kota.join(
    df_target,
    on="kota",
    how="inner"
)

hasil_A = hasil_A.withColumn(
    "pencapaian_persen",
    col("total_pendapatan") / col("target_bulanan") * 100
)

hasil_A.orderBy(
    col("pencapaian_persen").desc()
).show()

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



Tugas B - Window Function: Kategori Terlaris per Kota

In [19]:
pendapatan_kategori = df_transaksi.groupBy(
    "kota",
    "kategori"
).agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

window_kategori = Window \
    .partitionBy("kota") \
    .orderBy(col("total_pendapatan").desc())

hasil_B = pendapatan_kategori.withColumn(
    "rangking",
    row_number().over(window_kategori)
)

hasil_B.filter(
    col("rangking") == 1
).orderBy("kota").show()

+----------+--------------------+----------------+--------+
|      kota|            kategori|total_pendapatan|rangking|
+----------+--------------------+----------------+--------+
|  Magelang|Kesehatan & Kecan...|         7275000|       1|
| Purworejo|Kesehatan & Kecan...|        10075000|       1|
|  Semarang|        Rumah Tangga|        11125000|       1|
|      Solo|Kesehatan & Kecan...|         8425000|       1|
|Yogyakarta|             Fashion|        13325000|       1|
+----------+--------------------+----------------+--------+



Tugas C - Spark SQL

In [21]:
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target_cabang")

Selanjutnya digunakan Spark SQL untuk menghitung jumlah transaksi pada setiap kota.

In [22]:
hasil_C = spark.sql("""
    SELECT
        t.kota,
        d.pic_cabang,
        COUNT(t.order_id) AS jumlah_transaksi
    FROM transaksi t
    JOIN target_cabang d
        ON t.kota = d.kota
    GROUP BY
        t.kota,
        d.pic_cabang
    ORDER BY
        jumlah_transaksi DESC
""")

hasil_C.show()

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



Tugas D - Kesimpulan Analisis

Berdasarkan hasil pengolahan data pada bagian A dan B, terdapat perbedaan performa antara setiap cabang. Purworejo memiliki total pendapatan sebesar Rp45.650.000 dengan target bulanan Rp30.000.000, sehingga pencapaian targetnya mencapai 152,17%. Selain itu, Purworejo memiliki jumlah transaksi terbanyak, yaitu 116 transaksi, dan kategori dengan pendapatan tertinggi adalah Kesehatan & Kecantikan dengan pendapatan sebesar Rp10.075.000.

Di sisi lain, Semarang memiliki pencapaian target sebesar 69,41%, dengan total pendapatan Rp38.175.000 dari target Rp55.000.000. Jumlah transaksinya adalah 93 transaksi, sedangkan kategori dengan pendapatan tertinggi adalah Rumah Tangga sebesar Rp11.125.000. Berdasarkan angka tersebut, data menunjukkan bahwa Purworejo telah melampaui target bulanannya, sedangkan Semarang masih memiliki selisih yang cukup besar terhadap target. Data kategori juga dapat digunakan manajemen sebagai bahan untuk melihat jenis produk yang menghasilkan pendapatan terbesar pada masing-masing cabang.